In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Compute Graphs and Compilation

In that section the activation function `gelu_ish` took eight
to nine times as long as a single addition on the same tensor, although its
arithmetic hardly mattered. The cause was the way the framework ran it.
Each of its eight elementwise operations ran as a GPU kernel of its own,
which read its inputs from GPU memory and wrote its result back, and
that section showed why that memory traffic is slow. Can a
framework run such a chain without the traffic, and without the cost of
issuing every operation from Python?

A count of the bytes shows how much is wasted. The function computes
$0.5\, x \left(1 + \tanh\left(0.8\, (x + 0.04\, x^3)\right)\right)$.
Applied to a $4000 \times 4000$ matrix in fp32, which takes 64 MB, the
eight kernels together move 1.15 GB to produce 64 MB of output. Reading
the input once and writing the result once would move 128 MB, nine times
less. At the RTX 4090's memory bandwidth of about 1 TB/s, the eight
kernels need at least 1.15 ms where 0.13 ms would do.

Activation functions, normalization layers, residual additions, and
optimizer updates are all chains of this kind, and a deep network contains
hundreds of them. Small tensors add a second cost: launching a kernel from
Python takes longer than running it, and the GPU waits. Both costs come
from *eager execution*, which runs a program one operation at a time, as
Python reaches it. The remedy is to give the whole program to a compiler
before it runs. A compiler that sees all eight operations can merge them
into one kernel that reads $x$ once and writes the result once. A sequence
of kernels, once recorded, can be replayed without Python.

We study how a framework captures a program as a graph, what a compiler
does with that graph, and which programs escape capture.
Compilation addresses two of the three regimes of that section,
the bandwidth-bound and the overhead-bound regime. Memory capacity is a
separate limit, the subject of that section.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## Graph Capture

A compiler must see a program whole before it can rewrite it, but Python
executes a program one line at a time. A framework therefore has to obtain
the program as a graph first, and some Python constructs do not fit into a
graph.

### Computation Graphs

The computation of a neural network forms a *computation graph*: its nodes
are operations, and its edges carry tensors from one operation to the next.
Autograd records this graph during the forward pass so that it can walk it
backward to compute gradients (that section).
the figure shows the graph of a two-layer network
$\textrm{relu}(\mathbf{X}\mathbf{W}_1 + \mathbf{b}_1)\mathbf{W}_2 +
\mathbf{b}_2$ followed by a loss.

![The computation graph of a two-layer network with a loss. Autograd
records it to run the backward pass; a compiler that has the whole graph
before the forward pass can rewrite it.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-compute-graph.svg)

Autograd's record cannot speed up the forward pass. It is written while the
operations execute, so by the time it exists every operation has already run
as its own kernel. Each kernel needs a launch, which takes 1–5 µs in the
CUDA driver and about 5–15 µs of host time when Python dispatches the
operation (that section), and each elementwise kernel makes a
round trip through GPU memory. To merge the `add` with the `relu` that follows it, a
compiler needs the graph before the forward pass runs. Obtaining the graph
in advance is called *graph capture*.

### Tracing

The usual way to capture a graph is to run the function once on
*placeholders*, objects that carry the shape and dtype of a tensor but no
values. Every tensor operation applied to a placeholder is recorded instead
of computed, and the record is the graph. Python code that does not depend
on tensor values runs as usual and leaves only its effect: a loop over four
layers leaves four copies of the layer's operations, and arithmetic on
shapes leaves constants. Running a function to record its operations is
called *tracing*. A compiler then translates the graph into kernels, fusing
operations where it can, and caches them. Later calls with arguments of the
same shapes and dtypes reuse the cached kernels
(the figure).

![Graph capture. The first call traces the function on placeholders; a
compiler turns the recorded graph into kernels, which later calls with the
same shapes and dtypes reuse. A branch on a tensor's value cannot be
recorded, and a new shape or dtype starts a new
capture.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-compile-pipelines.svg)

In JAX, `jax.jit` wraps a function. At the first call it traces the
function with abstract *tracer* values as placeholders and records the
operations in a typed intermediate representation, the *jaxpr*
[@Frostig.Johnson.Leary.2018]. The XLA compiler turns the jaxpr into
fused kernels. The compiled code is cached under the shapes and dtypes of
the arguments, and `jax.make_jaxpr` prints the jaxpr of a function.

### Branches on Values

Tracing cannot follow a branch that depends on a value. A placeholder has a
shape but no value, so when the function asks `if x.sum() > 0:`, tracing
has no answer, and the answer may differ from call to call. A graph cannot
hold Python control flow, which leaves two options. The framework can
return the decision to Python at run time, between two graphs. Or the
program can express the branch as a tensor operation that the graph can
hold: a *select* computes both alternatives and keeps one, and a
*conditional* operation computes only the one it needs. A branch on a shape,
such as `if x.shape[0] > 4:`, causes no difficulty, since shapes are known
during tracing. We capture a function that branches on the sign of a sum:

In [ ]:
def f(x):
    x = x * 2
    if x.sum() > 0:        # Python needs the value of this comparison
        return x + 1
    return x - 1

try:
    jax.jit(f)(jnp.ones(8))
except jax.errors.TracerBoolConversionError as e:
    print(type(e).__name__)

def f_select(x):  # Both alternatives in the graph; select keeps one
    x = x * 2
    return jax.lax.select(x.sum() > 0, x + 1, x - 1)

print(jax.make_jaxpr(f_select)(jnp.ones(8)))

`jax.jit` does not return decisions to Python. Converting a tracer to a
Python `bool` raises an error, and the branch must be written as a tensor
operation. `jax.lax.select` computes both alternatives and keeps one, so the
whole function becomes a single graph: the jaxpr shows the comparison `gt`
feeding `select_n`. `jax.lax.cond` executes only one alternative, which
matters when they are expensive.

### Shapes and Recompilation

A compiler specializes the kernels to the shapes in the graph: it chooses
tile sizes, loop bounds, and memory layouts for them. A call with tensors
of a new shape or dtype therefore needs a new compilation. A training loop
whose last minibatch is smaller than the others, as with the Fashion-MNIST
loaders of that section, compiles once more for that
minibatch. A loop over sequences of varying length may compile at almost
every step. There are two remedies: keep the set of shapes small, for
instance by padding sequences to a few fixed lengths, or compile code that
works for a range of sizes. We call a compiled function on vectors of
lengths 8, 8, 16, and 32 and count its compilations:

In [ ]:
@jax.jit
def g(x):
    print(f'tracing g for shape {x.shape}')  # Python runs only while tracing
    return jnp.sin(x) + 1

for n in (8, 8, 16, 32):
    g(jnp.ones(n))

The `print` runs only while the function is traced, so it reports each
trace: one for length 8, none for the second call at length 8, and one each
for 16 and 32. Every new shape means a new trace and a new XLA compilation.
Other Python side effects, such as appending to a list or drawing from
NumPy's random generator, also happen at trace time only, and the compiled
function repeats none of them. An argument that is a Python value rather
than an array, such as a flag or a number of layers, can be marked static
with `static_argnums`; the function is then traced once for each distinct
value.

## Operation Fusion

Merging several operations into one kernel is called *fusion*. Its gain
can be predicted before anything runs, by counting the bytes that each
version moves through memory.

### Counting Memory Traffic

Python evaluates `gelu_ish(x)` as eight operations, each of which produces
a temporary array: $t_1 = 0.5x$, $t_2 = x^3$, $t_3 = 0.04\, t_2$,
$t_4 = x + t_3$, $t_5 = 0.8\, t_4$, $t_6 = \tanh t_5$, $t_7 = 1 + t_6$,
and the result $t_1 t_7$. In eager execution each operation is a kernel that
reads its array arguments from GPU memory and writes its result back. Six
operations read one array and two read two, so the chain reads ten arrays
and writes eight. A fused kernel reads $x$ once, keeps every temporary in
registers, and writes the result once (the figure). For arrays of
$n$ elements of $b$ bytes each, the ratio of the bytes moved is

$$\frac{\textrm{bytes moved, eager}}{\textrm{bytes moved, fused}} =
\frac{(10 + 8)\, n b}{(1 + 1)\, n b} = 9.$$

Both versions perform the same arithmetic, about a dozen FLOPs per element,
which is far less than the GPU can do while it moves the element's bytes.
Both are therefore limited by bandwidth, and fusion should shorten the
chain by the ratio of the bytes moved, a factor of nine.

![Memory traffic of `gelu_ish` on a 64 MB array. Eager execution runs eight
kernels that read ten arrays and write eight; the fused kernel reads the
input once, keeps the seven intermediate results in registers, and writes
the result once.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-perf-fusion.svg)

We compile `d2l.gelu_ish`, the function of that section, check
that the compiled version computes the same values, and time both versions. Each time is then converted into a
bandwidth: the bytes that version moves, divided by its time.

In [ ]:
x = jax.random.normal(d2l.get_key(), (4000, 4000))
compiled = jax.jit(d2l.gelu_ish)
assert jnp.allclose(d2l.gelu_ish(x), compiled(x), atol=1e-6)  # Same answer
arrays_moved = {'eager': 18, 'compiled': 2}  # In units of x.nbytes
for desc, fn in (('eager', d2l.gelu_ish), ('compiled', compiled)):
    b = d2l.Benchmark(lambda: fn(x), desc=desc)
    print(f'{b}, {arrays_moved[desc] * x.nbytes / b.time / 1e9:.0f} GB/s')

The compiled chain is eight to ten times faster, close to the predicted
nine, and both
versions move data at a similar rate, most of the card's
1 TB/s. The compiler made memory no faster and arithmetic no cheaper. It
removed 16 of the 18 array transfers. The cell checks the result before
timing it, because a faster kernel is of no use if it computes different
values. It uses `allclose`, since the compiler may reorder floating-point
operations: the results agree to rounding error, not bit for bit.

The same count applies at the scale of a language model. The chapter's
running example is a GPT with a billion parameters and width $d = 2048$,
trained on minibatches of 8 sequences of 1,024 tokens
(that section). Give it a GELU MLP of hidden width
$4d = 8192$ (`act='gelu'` in `d2l.GPT`). The activation then receives
$8192 \cdot 8192 \approx 6.7 \times 10^7$ elements per layer, 134 MB per
array in bf16. Written as `gelu_ish`, the eager chain moves 18 such arrays,
2.4 GB, which takes about 2.4 ms at 1 TB/s; the fused chain takes 0.27 ms.
The two matrix multiplications around the activation perform $2 \cdot 2 \cdot 8192 \cdot 2048 \cdot 8192 \approx
5.5 \times 10^{11}$ FLOPs, about 3.3 ms at the card's bf16 peak of
165 TFLOP/s. Unfused, the activation adds about three quarters to the time
of the MLP; fused, it adds under a tenth. Both times are lower bounds, since
real kernels reach neither peak.

The ratio of the two times does not depend on the number of tokens $M$.
The eager chain moves $18 \cdot 4dM$ elements of 2 bytes, and the two
matrix multiplications perform $2 \cdot 2 \cdot M \cdot d \cdot 4d$
FLOPs, so

$$\frac{t_\textrm{activation}}{t_\textrm{matmul}} =
\frac{144\, dM/\beta}{16\, d^2 M/P} = \frac{9 P}{\beta d}
\approx 0.72$$

for $P = 165$ TFLOP/s, $\beta = 1$ TB/s, and $d = 2048$. The unfused
activation matters less in wider models, since the arithmetic grows with
$d^2$ and the memory traffic only with $d$.

Real models seldom write GELU as a chain of eight operations. Libraries
ship hand-fused kernels for common activations, and `act='gelu'` calls one.
A compiler, however, fuses any chain, including ones that no library
anticipated: a SwiGLU gate, a custom activation, an optimizer update.

### Fusion Patterns

Compilers fuse three kinds of patterns. The simplest is the *elementwise
chain*, such as `gelu_ish`: a sequence of operations each of which computes
element $i$ of its output from element $i$ of its inputs. The whole chain
runs in one kernel in which each thread applies every operation to its own
elements.

A *reduction*, such as a softmax or a layer normalization, combines
elementwise operations with a maximum or a sum over each row. A fused kernel
loads a row into on-chip memory once, computes the reduction and the
normalized outputs there, and writes the row once.

An *epilogue* is a run of elementwise operations that follows a matrix
multiplication, such as adding a bias and applying an activation. The
compiler applies them to each tile of the product while the tile is still in
registers, before it is written to memory.

Compilers do not change the algorithm. A large matrix multiplication is
limited by arithmetic and already runs as a single tuned library kernel;
compiling the code around it leaves it as it is. Nor can fusion remove an
intermediate result that the algorithm needs in full. Attention computes a
$T \times T$ matrix of scores, normalizes each row with a softmax, and
multiplies the result by the values. A compiler fuses the softmax but still
writes the score matrix to memory. FlashAttention
(that section) never writes it: it splits the
computation into tiles and updates the softmax incrementally as it goes
[@Dao.Fu.Ermon.ea.2022], a change of algorithm that general-purpose
compilers do not find. Such kernels are written by hand, often in Triton
[@Tillet.Kung.Cox.2019], a language for GPU kernels embedded in
Python.

## Compiling a Training Step

A training step contains three kinds of work: matrix multiplications in
the forward and backward passes, elementwise chains for the activations and
their derivatives, and an elementwise update of every parameter. Compiling
the whole step lets the compiler fuse the elementwise work of all three,
while the matrix multiplications stay library kernels. Compilation itself
takes time, from seconds for a small step to minutes for a large model,
and that time is spent at the first call. If compilation takes
$t_\textrm{compile}$ and the compiled step saves
$t_\textrm{eager} - t_\textrm{compiled}$ per call, the compiled run
overtakes the eager one after

$$n^* = \frac{t_\textrm{compile}}{t_\textrm{eager} - t_\textrm{compiled}}$$

steps. A run much longer than $n^*$ steps gains from compilation, and a
shorter one loses. With a compilation time of 30 s and a saving of 10 ms
per step, for instance, the break-even point is 3,000 steps, a fraction of
a typical training run but more than most quick experiments.

Our step trains a multilayer perceptron with four linear layers of width
1,024 and `gelu_ish` between them. It computes the squared loss of the
network's outputs against targets, then the gradients, and updates the
parameters by gradient descent.

In [ ]:
def init_mlp(key, width=1024, depth=4):
    """Weights and biases drawn uniformly from +-1/sqrt(width)."""
    bound = 1 / width**0.5
    keys = jax.random.split(key, (depth, 2))
    return [(jax.random.uniform(kW, (width, width), minval=-bound,
                                maxval=bound),
             jax.random.uniform(kb, (width,), minval=-bound, maxval=bound))
            for kW, kb in keys]

def mlp(params, X):  # Linear layers with the gelu_ish chain between them
    for W, b in params[:-1]:
        X = d2l.gelu_ish(X @ W + b)
    W, b = params[-1]
    return X @ W + b

def train_step(params, X, Y, lr=0.1):  # Forward, backward, SGD update
    loss_fn = lambda p: ((mlp(p, X) - Y) ** 2).mean()
    loss, grads = jax.value_and_grad(loss_fn)(params)
    return loss, jax.tree.map(lambda p, g: p - lr * g, params, grads)

The step is a pure function: it takes the parameters and the data and
returns the loss and the updated parameters. The weights and biases are
drawn uniformly from $[-1/\sqrt{1024}, 1/\sqrt{1024}]$.

We compile the step and time its first call, which includes the
compilation. Then we time the eager and the compiled step and compute $n^*$
from the equation:

In [ ]:
params = init_mlp(d2l.get_key())
X, Y = (jax.random.normal(d2l.get_key(), (512, 1024)) for _ in range(2))
compiled_step = jax.jit(train_step)
t_first = d2l.Benchmark(lambda: compiled_step(params, X, Y),
                        warmup=0, repeats=1).time  # Includes compilation
t_eager = d2l.Benchmark(lambda: train_step(params, X, Y)).time
t_compiled = d2l.Benchmark(lambda: compiled_step(params, X, Y)).time
print(f'first call: {t_first:.1f} s\n'
      f'eager: {1000 * t_eager:.2f} ms, compiled: {1000 * t_compiled:.2f} ms\n'
      f'break-even: {t_first / (t_eager - t_compiled):.0f} steps')

The first call takes far longer than a step, up to several seconds, and
the printed $n^*$ is a small fraction of the steps of a real training run.

`jax.jit(train_step)` compiles the loss, the gradients, and the update into
one XLA program at its first call. The compiler can also run *ahead of
time*, before any call: `jax.jit(train_step).lower(params, X, Y).compile()`
returns the compiled executable, whose `memory_analysis()` reports the
memory the compiler plans to use (that section reads
it). The un-jitted step is about two orders of magnitude slower than the
compiled one, because each operation of the forward pass, the backward
pass, and the update is then dispatched from Python on its own, at tens of
microseconds per dispatch. Un-jitted JAX is for debugging; training steps
are jitted.

## Kernel Replay

Fusion reduces memory traffic, but it cannot merge matrix multiplications,
and each one still needs a launch. Consider a stack of 60 linear layers of
width 256 without biases, each followed by a tanh, applied to a minibatch
of 64. Each layer performs a matrix multiplication of
$2 \cdot 64 \cdot 256 \cdot 256 \approx 8.4 \times 10^6$ FLOPs, which the
4090 completes in a few microseconds, and a tanh over 16,384 elements,
which takes about as long. Eager execution launches 120 kernels per call.
Python spends 5–15 µs of host time issuing a bare operation
(that section), longer than the GPU needs to run it. The GPU
therefore finishes each kernel before Python has issued the next one, and
waits. The program is overhead-bound. Fusing the tanh into the preceding matrix
multiplication would halve the number of launches, but the remaining 60
would still keep the GPU waiting.

The remedy is to take Python out of the loop: record the sequence of kernel
launches once, and replay the whole sequence with a single command. NVIDIA
GPUs support this through *CUDA graphs*. A CUDA graph stores each kernel
together with its launch configuration and the memory addresses of its
arguments. Replaying it takes one launch from the host, after which the GPU
runs the kernels back to back. The recording is rigid: its shapes and
memory addresses are fixed. New inputs must therefore be copied into the
recorded input buffers before a replay, and the outputs of one replay are
overwritten by the next. Nothing inside the sequence may wait for a value
read by the CPU.

In [ ]:
def init_deep(key, depth=60, width=256):  # 120 small kernels per call
    return list(jax.random.uniform(key, (depth, width, width),
                                   minval=-width**-0.5, maxval=width**-0.5))

def deep(params, x):
    for W in params:
        x = jnp.tanh(x @ W)
    return x

params = init_deep(d2l.get_key())
x = jax.random.normal(d2l.get_key(), (64, 256))
jitted = jax.jit(deep)
jitted(params, x)  # First call compiles
print(d2l.Benchmark(lambda: deep(params, x), desc='eager'))
print(d2l.Benchmark(lambda: jitted(params, x), desc='jit'))

Without Python in the loop, the stack runs many times faster, at
roughly the time the GPU needs for its 120 small kernels. Dividing each time
by 120 reads off the two costs per kernel: the eager time gives the host
time of one launch, and the compiled time gives the GPU time of one kernel,
a few microseconds.

`jax.jit` already removes Python from the loop: the compiled function
starts with one call from Python. The eager version dispatches its 120
operations one at a time, at tens of microseconds each, and is twenty to
thirty times slower. On a GPU, XLA also records runs of consecutive kernels as
CUDA graphs by default (XLA calls them command buffers). A jitted function
is therefore recorded and replayed without a separate mode.

## Limits of Compilation

Compilation speeds up code that runs many times with the same shapes and is
limited by bandwidth or overhead. Outside these conditions it can take more
time than it saves.

* **Short runs.** A function called fewer than the $n^*$ times of
  the equation loses time. Quick experiments, evaluation
  over small datasets, and functions called once are usually faster eager.
* **Branches on values.** A hot loop full of branches on tensor values is
  captured, if at all, in fragments separated by returns to Python. It
  gains from compilation only after its branches are rewritten as selects
  or conditional operations.
* **Changing shapes.** Each new shape compiles again. Pad to a few fixed
  sizes, compile for symbolic sizes, or leave the function eager.
* **Compute-bound code.** If a profile shows one large matrix
  multiplication taking most of the time, there is nothing to fuse and no
  launch overhead to remove. Compiling rarely hurts such code, but it cannot
  speed it up much.
* **Debugging.** Compiled code does not run line by line. Breakpoints and
  print statements inside it behave differently, and errors point into
  generated code. Debug eagerly, then compile.

Compilation removes memory traffic and launches from a program that fits on
one GPU. It leaves the largest consumers of memory where they are: the
parameters, the optimizer state, and the activations saved for the backward
pass. that section measures them and shows how lower
precision and recomputation reduce them.

## Summary

Eager execution runs a program one operation at a time, so every elementwise
operation makes a round trip through GPU memory and every operation needs a
launch from Python. Graph capture traces the program on placeholders and
records its tensor operations as a graph specialized to the shapes it saw; a
branch on a tensor value cannot be recorded, and a new shape requires a new
compilation. A compiler fuses elementwise chains, reductions, and the
epilogues of matrix multiplications into single kernels, which for
`gelu_ish` divides the memory traffic, and with it the time, by about nine.
Recording a sequence of kernels as a CUDA graph and replaying it removes the
launch overhead of stacks of small kernels. Compilation takes time at the
first call, so it saves time only for code that runs often, with fixed
shapes, in the bandwidth or overhead regime; it cannot speed up a
compute-bound matrix multiplication.

## Exercises

1. [code] **Value-dependent branches.** Tracing replaces the values of the
   arguments by placeholders (that section).
    1. Rewrite the branch of `f` once with `jnp.where` and once with
       `jax.lax.cond`, and print both jaxprs with `jax.make_jaxpr`. Which
       operations does each version execute at run time?
    1. Apply `jax.vmap` to the `jax.lax.cond` version over a batch of
       vectors and print its jaxpr. Explain what became of the conditional,
       and why.
    1. Jit a function that returns `x + 1` when `x.shape[0] > 4` and
       `x - 1` otherwise, and call it on vectors of lengths 3, 5, and 6.
       Explain why it does not raise the error that `f` raises, and count
       its traces.

2. [code] **Shape changes.** A jitted function is specialized to the shapes
   of its arguments (that section).
    1. Jit a function that returns the mean of `jnp.tanh(x)` over a vector
       `x`, and let its Python body increment a global counter. Call it in
       a loop on vectors of lengths 100, 200, and 300, three times each.
       Count the traces, and time the first and the later calls at each
       length.
    1. Pad every input to length 512 and pass the function what it needs
       to exclude the padding from the mean, so that one trace serves all
       three lengths. Check the results against the unpadded version.
    1. Instead, pass the padded vector together with its length, mark the
       length with `static_argnums`, and slice inside the function. Count
       the traces and explain the count. Name an argument of a training
       step for which `static_argnums` is the right tool.

3. **Padding buckets.** A compiled step that is specialized to static
   shapes (that section) compiles once for every distinct
   input shape it sees. Consider a training run of $10^5$ steps in which
   every batch holds sequences of one length $\ell$, drawn uniformly from
   $\{1, \ldots, 512\}$, and is padded to a length $\ell' \geq \ell$. One
   compilation costs 20 s, and a step at padded length $\ell'$ costs
   $50\,\textrm{ms} \cdot \ell'/512$.
    1. Pad every batch to the next multiple of $w$, for $w$ dividing 512.
       Find the number of compilations and the expected padded length as
       functions of $w$, and write the total run time $T(w)$.
    1. Minimize $T(w)$ over $w$ treated as a continuous variable, and state
       how the optimal $w$ scales with the compilation cost and with the
       number of steps. Compare $T$ at $w = 1$, at the two powers of two
       nearest the optimum, and at $w = 512$.
    1. Padding to the next power of two compiles ten shapes. Compute its
       expected padded length and total run time, and compare them with
       those of $w = 64$. For which distribution of lengths would powers
       of two be the better choice?

1. [code] **Fusion behind a matmul.** ● Compilation leaves a large matrix
   product unchanged but fuses the elementwise chain that follows it.
   Consider `gelu_ish(a @ a)` for an $n \times n$ matrix `a` in bf16 on a
   GPU with peak compute $P$, bandwidth $\beta$, and ridge point
   $I^* = P/\beta$, the arithmetic intensity at which the time to compute
   equals the time to move the data. Model the product as compute-bound,
   taking $2n^3/P$ seconds. Model the chain as bandwidth-bound: eager
   execution moves $m$ arrays of $n^2$ elements of $b$ bytes each through
   memory, and the fused kernel moves two such arrays.
    1. Derive the speedup bound $S(n)$ of the compiled over the eager
       computation in terms of $n$, $m$, $b$, and $I^*$, and show that it
       tends to 1 as $n$ grows. With $m = 18$ as counted for `gelu_ish`,
       evaluate $S(n)$ for bf16 on an RTX 4090 ($I^* \approx 165$
       FLOP/byte) at $n = 1024$, $4096$, and $16384$, and find the $n$ at
       which $S(n)$ falls to $1.1$.
    1. Measure the eager and the compiled time of `gelu_ish(a @ a)` with
       `d2l.Benchmark` for $n = 256, 512, \ldots, 8192$. Plot the measured
       speedup together with $S(n)$, and explain where and why the two
       differ.

1. **Fusing a reduction.** ● A numerically safe softmax over each row of an
   $R \times d$ fp32 matrix can be written as five eager operations: the
   row maximum, the subtraction of that maximum, the exponential, the row
   sum, and the division by that sum.
    1. Count the elements that the five eager kernels read from and write
       to memory, keeping only terms proportional to $Rd$. Count the same
       for a single fused kernel that holds each row in on-chip memory, and
       state the speedup bound for a bandwidth-bound softmax.
    1. A row longer than the on-chip memory must be read from memory more
       than once. Find the smallest number of passes over a row, and the
       resulting traffic, for a fused kernel that computes the maximum and
       the sum in separate passes. Then do the same for a kernel that
       maintains both in one pass over the entries $x_1, \ldots, x_d$ of a
       row, with $m_0 = -\infty$, $s_0 = 0$, and
       $m_j = \max(m_{j-1}, x_j)$,
       $s_j = s_{j-1}\, e^{m_{j-1} - m_j} + e^{x_j - m_j}$. Check that
       $s_d = \sum_j e^{x_j - m_d}$.

6. [code] **Trace-time constants.** The Python body of a jitted function
    runs only while it is traced (that section).
    1. Jit a function that returns `x + np.random.randn()`, call it twice,
       and print its jaxpr. Explain the two outputs.
    1. A dropout function draws its mask with `jax.random.bernoulli` from
       a key held in a global variable. Jit it, call it, assign a new key
       to the global variable, and call it again. Explain the result, and
       rewrite the function so that every step draws a fresh mask.

7. [code] **Overhead crossover.** ● Use the stack of 60 layers of width
   256 at batch size 64 from that section, with its eager
   and its compiled version. Compilation removes the host time of each
   launch, about 5–15 µs, from the critical path, but not the time each
   kernel takes to execute.
    1. Time both versions for depths $L$ from 10 to 200 at width 256, and
       fit each time as $t(L) = c + sL$. Compare the eager slope per kernel
       with the launch cost, and explain the compiled slope.
    1. A kernel that performs $F$ FLOPs and moves $B$ bytes takes at least
       $\max(F/P, B/\beta)$ seconds on a GPU with peak compute $P$ and
       bandwidth $\beta$. Predict from this bound the width at which the
       matrix product of one layer takes as long to execute as to launch.
       Then time both versions at depth 60 for widths from 64 to 4096,
       plot the ratio of eager to compiled time against the width, and
       compare the width at which the ratio approaches 1 with your
       prediction.

1. [extended] **A tracer of your own.** ●● Build a miniature tracing
   compiler in plain Python, with NumPy arrays as the values on which the
   captured program finally runs.
    1. Write a class `Tracer` whose instances carry a `shape` but no
       values, and functions `tanh`, `exp`, and `relu` that take a
       `Tracer`. Each operation on a `Tracer` appends a node to a shared
       graph and returns a new `Tracer`. A node holds the operation, its
       inputs, and the shape of its result. Support the arithmetic
       operators, the matrix product `@`, the comparison `>`, the methods
       `sum` and `max` with `axis` and `keepdims` arguments, and the three
       functions. Trace a copy of `gelu_ish` that calls your `tanh` on a
       placeholder of a given shape, and print the graph one node per line.
    1. Make `Tracer.__bool__` raise an error. Trace a function that
       returns `x + 1` when `x.shape[0] > 4` and `x - 1` otherwise, and a
       second function that branches the same way on `x.sum() > 0`.
       Explain which of the two your tracer can record, and what the
       recorded graph keeps of the branch.
    1. Write an interpreter that executes a graph node by node on NumPy
       arrays, and check its result against a direct NumPy evaluation of
       `gelu_ish`.
    1. Write a pass that partitions the graph into fused kernels under the
       rule that an elementwise node may join the kernel of its inputs, a
       matrix product forms a kernel of its own, and a reduction ends the
       kernel it belongs to. State how your pass treats a node whose inputs
       come from different kernels. For `gelu_ish`, for the five-operation
       softmax, and for the two-layer network
       `relu(x @ W1 + b1) @ W2 + b2`, count the kernel launches and the
       array elements moved through memory under one kernel per node and
       under your partition.

[Discussions](https://d2l.discourse.group/t/36789)